In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, DoubleType, StringType, DateType
)

# --------------------------------------------------------------------------
# CONFIG — edit these two lines for your setup
# --------------------------------------------------------------------------
CSV_PATH = "/content/Wildfire_Dataset.csv"        # path to your 1.1GB file
KB_OUTPUT_PATH = "wildfire_kb/enriched.parquet"

# --------------------------------------------------------------------------
# 1. Spark session
#    - driver memory bumped up since it's a 1.1GB file; adjust to your machine.
#    - shuffle partitions lowered from Spark's default 200 since this isn't a
#      huge cluster job; tune up if you're running on an actual cluster.
# --------------------------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("WildfireKnowledgeBase")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# --------------------------------------------------------------------------
# 2. Explicit schema — much faster and safer than inferSchema=True on 1GB+ files
# --------------------------------------------------------------------------
schema = StructType([
    StructField("latitude",  DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("datetime",  DateType(),   True),
    StructField("Wildfire",  StringType(), True),   # "Yes" / "No"
    StructField("pr",        DoubleType(), True),   # precipitation
    StructField("rmax",      DoubleType(), True),   # max relative humidity
    StructField("rmin",      DoubleType(), True),   # min relative humidity
    StructField("sph",       DoubleType(), True),   # specific humidity
    StructField("srad",      DoubleType(), True),   # solar radiation
    StructField("tmmn",      DoubleType(), True),   # min temperature (K)
    StructField("tmmx",      DoubleType(), True),   # max temperature (K)
    StructField("vs",        DoubleType(), True),   # wind speed
    StructField("bi",        DoubleType(), True),   # burning index
    StructField("fm100",     DoubleType(), True),   # 100-hr fuel moisture
    StructField("fm1000",    DoubleType(), True),   # 1000-hr fuel moisture
    StructField("erc",       DoubleType(), True),   # energy release component
    StructField("etr",       DoubleType(), True),   # reference evapotranspiration
    StructField("pet",       DoubleType(), True),   # potential evapotranspiration
    StructField("vpd",       DoubleType(), True),   # vapor pressure deficit
])

raw = spark.read.csv(CSV_PATH, header=True, schema=schema)
raw.createOrReplaceTempView("wildfire_raw")

print(f"Raw row count: {raw.count():,}")
raw.printSchema()

# --------------------------------------------------------------------------
# 3. Clean: drop rows missing coordinates/date/label; drop obviously invalid
#    sensor values (negative humidity/precip, etc. — adjust bounds as you
#    inspect your actual data's edge cases).
# --------------------------------------------------------------------------
clean = spark.sql("""
    SELECT *
    FROM wildfire_raw
    WHERE latitude IS NOT NULL
      AND longitude IS NOT NULL
      AND datetime IS NOT NULL
      AND Wildfire IN ('Yes', 'No')
      AND pr >= 0
      AND rmin >= 0 AND rmax <= 100
""")
clean.createOrReplaceTempView("wildfire_clean")
print(f"Rows after cleaning: {clean.count():,}")

# --------------------------------------------------------------------------
# 4. Enrich: label, calendar features, location id
# --------------------------------------------------------------------------
enriched = spark.sql("""
    SELECT
        *,
        CASE WHEN Wildfire = 'Yes' THEN 1 ELSE 0 END AS label,
        YEAR(datetime)  AS year,
        MONTH(datetime) AS month,
        CASE
            WHEN MONTH(datetime) IN (12, 1, 2) THEN 'Winter'
            WHEN MONTH(datetime) IN (3, 4, 5)  THEN 'Spring'
            WHEN MONTH(datetime) IN (6, 7, 8)  THEN 'Summer'
            ELSE 'Fall'
        END AS season,
        CONCAT(CAST(latitude AS STRING), '_', CAST(longitude AS STRING)) AS location_id
    FROM wildfire_clean
""")

# 7-day trailing rolling weather features per location (fire risk builds up
# over days of hot/dry/windy weather, not just on a single day's reading)
loc_window = Window.partitionBy("location_id").orderBy("datetime").rowsBetween(-6, 0)

enriched = (
    enriched
    .withColumn("pr_7d_sum",    F.sum("pr").over(loc_window))
    .withColumn("tmmx_7d_avg",  F.avg("tmmx").over(loc_window))
    .withColumn("rmin_7d_avg",  F.avg("rmin").over(loc_window))
    .withColumn("vs_7d_avg",    F.avg("vs").over(loc_window))
    .withColumn("bi_7d_avg",    F.avg("bi").over(loc_window))
    .withColumn("erc_7d_avg",   F.avg("erc").over(loc_window))
    .withColumn("dry_streak_days",
        F.sum(F.when(F.col("pr") == 0, 1).otherwise(0)).over(loc_window))
)

enriched.createOrReplaceTempView("wildfire_kb")

# --------------------------------------------------------------------------
# 5. Persist the knowledge base to Parquet (fast columnar reads for Stage 2,
#    and for any BI/SQL tool you point at this later).
# --------------------------------------------------------------------------
(
    enriched.write
    .mode("overwrite")
    .partitionBy("year")
    .parquet(KB_OUTPUT_PATH)
)
print(f"Knowledge base written to: {KB_OUTPUT_PATH}")

# --------------------------------------------------------------------------
# 6. Example knowledge-base queries — this is the "explore with SparkSQL"
#    part. Add your own views here as you explore.
# --------------------------------------------------------------------------
print("\n=== Fire rate by season ===")
spark.sql("""
    SELECT season,
           COUNT(*) AS total_days,
           SUM(label) AS fire_days,
           ROUND(100.0 * SUM(label) / COUNT(*), 3) AS fire_rate_pct
    FROM wildfire_kb
    GROUP BY season
    ORDER BY fire_rate_pct DESC
""").show()

print("\n=== Fire rate by year ===")
spark.sql("""
    SELECT year,
           COUNT(*) AS total_days,
           SUM(label) AS fire_days,
           ROUND(100.0 * SUM(label) / COUNT(*), 3) AS fire_rate_pct
    FROM wildfire_kb
    GROUP BY year
    ORDER BY year
""").show()

print("\n=== Average weather conditions: fire days vs non-fire days ===")
spark.sql("""
    SELECT label,
           ROUND(AVG(tmmx), 2)  AS avg_max_temp_k,
           ROUND(AVG(rmin), 2)  AS avg_min_humidity,
           ROUND(AVG(vs), 2)    AS avg_wind_speed,
           ROUND(AVG(bi), 2)    AS avg_burning_index,
           ROUND(AVG(erc), 2)   AS avg_energy_release_component,
           ROUND(AVG(pr_7d_sum), 2) AS avg_7day_precip
    FROM wildfire_kb
    GROUP BY label
""").show()

print("\n=== Top 10 highest fire-rate locations (min 30 observed days) ===")
spark.sql("""
    SELECT location_id,
           COUNT(*) AS days_observed,
           SUM(label) AS fire_days,
           ROUND(100.0 * SUM(label) / COUNT(*), 2) AS fire_rate_pct
    FROM wildfire_kb
    GROUP BY location_id
    HAVING COUNT(*) >= 30
    ORDER BY fire_rate_pct DESC
    LIMIT 10
""").show(truncate=False)

spark.stop()
print("\nDone. Run 02_train_ml_models.py next.")

Raw row count: 8,304,943
root
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- datetime: date (nullable = true)
 |-- Wildfire: string (nullable = true)
 |-- pr: double (nullable = true)
 |-- rmax: double (nullable = true)
 |-- rmin: double (nullable = true)
 |-- sph: double (nullable = true)
 |-- srad: double (nullable = true)
 |-- tmmn: double (nullable = true)
 |-- tmmx: double (nullable = true)
 |-- vs: double (nullable = true)
 |-- bi: double (nullable = true)
 |-- fm100: double (nullable = true)
 |-- fm1000: double (nullable = true)
 |-- erc: double (nullable = true)
 |-- etr: double (nullable = true)
 |-- pet: double (nullable = true)
 |-- vpd: double (nullable = true)

Rows after cleaning: 8,282,893
Knowledge base written to: wildfire_kb/enriched.parquet

=== Fire rate by season ===
+------+----------+---------+-------------+
|season|total_days|fire_days|fire_rate_pct|
+------+----------+---------+-------------+
|  Fall|   1862124|   115464| 